# ClinicalGuard — Model Training Notebook
**Purpose:** Train and evaluate three ML models on the ClinicalGuard realistic fraud dataset.

**Models compared:** Decision Tree · Random Forest · XGBoost

**Primary metric:** MANIPULATED class recall (catching fraud > overall accuracy)

**Output files (download after training):**
- `dt_model.pkl` — Decision Tree
- `rf_model.pkl` — Random Forest  
- `xgb_model.pkl` — XGBoost
- `scaler.pkl` — MinMaxScaler (fitted on training data only)
- `feature_cols.json` — exact feature order used in training
- `evaluation_report.txt` — full metrics for all three models

---
**Instructions:**
1. Upload `clinicalguard_realistic_dataset.csv` when prompted (Step 2)
2. Run all cells top to bottom
3. Download the output files at the end
4. Replace `backend/models/` with the new pkl files

## Step 1 — Install dependencies

In [ ]:
!pip install -q scikit-learn xgboost pandas numpy matplotlib seaborn shap
print('All packages ready.')

## Step 2 — Upload dataset

In [ ]:
from google.colab import files
print('Upload clinicalguard_realistic_dataset.csv')
uploaded = files.upload()
DATASET_PATH = list(uploaded.keys())[0]
print(f'Using: {DATASET_PATH}')

## Step 3 — Load and explore dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json, pickle, warnings
warnings.filterwarnings('ignore')

df = pd.read_csv(DATASET_PATH)
print(f'Shape: {df.shape}')
print(f'\nClass distribution:')
print(df['integrity_label'].value_counts())
print(f'\nManipulation types:')
print(df['manipulation_type'].value_counts())
df.head(10)

In [ ]:
# Feature statistics per class
FEAT = ['age','bp_systolic','bp_diastolic','glucose','hr','spo2',
        'diagnosis_encoded','previous_trials','product_experience',
        'last_trial_outcome','health_risk_score','age_grp_adult','age_grp_elderly']

print('=== AUTHENTIC stats ===')
print(df[df['integrity_label']==1][FEAT].describe().round(2))
print('\n=== MANIPULATED stats ===')
print(df[df['integrity_label']==0][FEAT].describe().round(2))

In [ ]:
# Visualise key feature distributions
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
key_feats = ['health_risk_score','bp_systolic','bp_diastolic','glucose','hr','spo2']
colors = {1:'#14B8A6', 0:'#EF4444'}
labels = {1:'Authentic', 0:'Manipulated'}

for ax, feat in zip(axes.flat, key_feats):
    for lbl in [1, 0]:
        subset = df[df['integrity_label']==lbl][feat]
        ax.hist(subset, bins=40, alpha=0.6, color=colors[lbl], label=labels[lbl])
    ax.set_title(feat)
    ax.legend()

plt.suptitle('Feature Distributions: Authentic vs Manipulated', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: feature_distributions.png')

## Step 4 — Prepare features and split

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

TARGET = 'integrity_label'
X = df[FEAT]
y = df[TARGET].astype(int)

# Stratified 70/30 split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# Fit scaler on training data ONLY
scaler = MinMaxScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {len(X_train)} | Test: {len(X_test)}')
print(f'Train class dist: {dict(y_train.value_counts().sort_index())}')
print(f'Test  class dist: {dict(y_test.value_counts().sort_index())}')
print(f'Scaler bp_systolic max: {scaler.data_max_[FEAT.index("bp_systolic")]:.1f} (expect <= 170)')

## Step 5 — Train all three models

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (confusion_matrix, classification_report,
                              roc_auc_score, recall_score, precision_score,
                              f1_score, accuracy_score)

def evaluate(name, model, X_te, y_te):
    yp   = model.predict(X_te)
    ypr  = model.predict_proba(X_te)[:,1]
    acc  = accuracy_score(y_te, yp)
    roc  = roc_auc_score(y_te, ypr)
    pm   = precision_score(y_te, yp, pos_label=0, zero_division=0)
    rm   = recall_score(y_te,    yp, pos_label=0, zero_division=0)
    fm   = f1_score(y_te,        yp, pos_label=0, zero_division=0)
    pa   = precision_score(y_te, yp, pos_label=1, zero_division=0)
    ra   = recall_score(y_te,    yp, pos_label=1, zero_division=0)
    fa   = f1_score(y_te,        yp, pos_label=1, zero_division=0)
    cm   = confusion_matrix(y_te, yp, labels=[0,1])
    print(f'\n{"="*55}')
    print(f'  {name}')
    print(f'{"="*55}')
    print(f'  Accuracy : {acc*100:.2f}%   ROC-AUC: {roc:.4f}')
    print(f'  MANIPULATED  Prec:{pm*100:6.1f}%  Recall:{rm*100:6.1f}%  F1:{fm*100:6.1f}%')
    print(f'  AUTHENTIC    Prec:{pa*100:6.1f}%  Recall:{ra*100:6.1f}%  F1:{fa*100:6.1f}%')
    print(f'  Confusion Matrix [0=MANIP, 1=AUTH]:')
    print(f'    TN:{cm[0,0]}  FP:{cm[0,1]}  FN:{cm[1,0]}  TP:{cm[1,1]}')
    print(f'    FP (MANIP→AUTH, dangerous): {cm[0,1]}')
    status = 'PASS' if rm >= 0.85 else 'FAIL'
    print(f'  [{status}] MANIPULATED recall >= 85% threshold')
    return {'name':name,'acc':acc,'roc':roc,'prec_m':pm,'rec_m':rm,'f1_m':fm,
            'prec_a':pa,'rec_a':ra,'f1_a':fa,'fp':cm[0,1],'fn':cm[1,0]}

results = {}

# --- Decision Tree ---
dt = DecisionTreeClassifier(
    criterion='gini', max_depth=8, min_samples_leaf=2,
    min_samples_split=4, class_weight='balanced', random_state=42
)
dt.fit(X_train_sc, y_train)
results['dt'] = evaluate('Decision Tree (DT)', dt, X_test_sc, y_test)
print(f'  Depth: {dt.get_depth()} | Leaves: {dt.get_n_leaves()}')

# --- Random Forest ---
rf = RandomForestClassifier(
    n_estimators=200, max_depth=10, min_samples_leaf=2,
    class_weight='balanced', n_jobs=-1, random_state=42
)
rf.fit(X_train_sc, y_train)
results['rf'] = evaluate('Random Forest (RF)', rf, X_test_sc, y_test)

# --- XGBoost ---
neg = (y_train==1).sum(); pos = (y_train==0).sum()
xgb = XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    scale_pos_weight=neg/pos,  # handles class imbalance
    use_label_encoder=False, eval_metric='logloss',
    n_jobs=-1, random_state=42
)
xgb.fit(X_train_sc, y_train)
results['xgb'] = evaluate('XGBoost (XGB)', xgb, X_test_sc, y_test)

## Step 6 — Model comparison table

In [ ]:
comp = pd.DataFrame(results).T
comp.index.name = 'Model'
display_cols = ['acc','roc','rec_m','prec_m','f1_m','rec_a','fp','fn']
rename = {'acc':'Accuracy','roc':'ROC-AUC','rec_m':'MANIP Recall',
          'prec_m':'MANIP Prec','f1_m':'MANIP F1',
          'rec_a':'AUTH Recall','fp':'FP (Dangerous)','fn':'FN (Nuisance)'}
comp_display = comp[display_cols].rename(columns=rename).round(4)
print('\n=== MODEL COMPARISON ===')
print(comp_display.to_string())

# Pick winner by MANIPULATED recall (primary), then ROC-AUC (secondary)
best_key = comp['rec_m'].astype(float).idxmax()
print(f'\nWINNER (best MANIPULATED recall): {results[best_key]["name"]}')

## Step 7 — Feature importance + SHAP (best model)

In [ ]:
import shap

# Feature importance from all models
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, (key, model, title) in zip(axes, [
    ('dt', dt, 'Decision Tree'),
    ('rf', rf, 'Random Forest'),
    ('xgb', xgb, 'XGBoost'),
]):
    imp = model.feature_importances_
    sorted_idx = np.argsort(imp)[::-1]
    ax.barh([FEAT[i] for i in sorted_idx], imp[sorted_idx], color='#3B82F6')
    ax.set_title(f'{title}\nFeature Importance')
    ax.invert_yaxis()

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: feature_importance.png')

In [ ]:
# SHAP on Random Forest (most stable for SHAP TreeExplainer)
print('Computing SHAP values on 500 test samples...')
explainer = shap.TreeExplainer(rf)
shap_vals  = explainer.shap_values(X_test_sc[:500])
# shap_vals is list [class0, class1] for RF
sv = shap_vals[0] if isinstance(shap_vals, list) else shap_vals
plt.figure(figsize=(10, 6))
shap.summary_plot(sv, X_test_sc[:500], feature_names=FEAT, show=False)
plt.title('SHAP Summary — MANIPULATED class (Random Forest)')
plt.tight_layout()
plt.savefig('shap_summary.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: shap_summary.png')

## Step 8 — Sanity checks on known records

In [ ]:
sanity = [
    ('Healthy patient (expect AUTHENTIC)',
     [35,115,75,90,68,99,4,1,2,1,0.12,1,0], 1),
    ('SFO case: high BP+glucose, HRS=0.05 (expect MANIPULATED)',
     [52,148,95,165,105,92,2,2,1,0,0.05,1,0], 0),
    ('VSM case: normal vitals but HRS=0.72 (expect MANIPULATED)',
     [45,132,85,118,94,96,1,1,1,1,0.72,1,0], 0),
    ('TSF case: high risk but outcome=1 (expect MANIPULATED)',
     [58,155,98,175,108,90,2,2,2,1,0.78,1,0], 0),
    ('Elderly authentic patient',
     [68,135,85,130,78,96,1,3,1,1,0.42,0,1], 1),
]

print(f'{"Case":<50} {"DT":>12} {"RF":>12} {"XGB":>12} {"Expected":>12}')
print('-'*100)
for desc, feats, expected in sanity:
    arr = np.array(feats).reshape(1,-1)
    arr_sc = scaler.transform(arr)
    exp_label = 'AUTHENTIC' if expected==1 else 'MANIPULATED'
    def pred_label(m):
        p = int(m.predict(arr_sc)[0])
        conf = m.predict_proba(arr_sc)[0][p]
        lbl = 'AUTH' if p==1 else 'MANIP'
        ok  = '✓' if p==expected else '✗'
        return f'{ok}{lbl}({conf*100:.0f}%)'
    print(f'{desc:<50} {pred_label(dt):>12} {pred_label(rf):>12} {pred_label(xgb):>12} {exp_label:>12}')

## Step 9 — Save all model artefacts

In [ ]:
import os

# Save all three models
with open('dt_model.pkl',  'wb') as f: pickle.dump(dt,  f)
with open('rf_model.pkl',  'wb') as f: pickle.dump(rf,  f)
with open('xgb_model.pkl', 'wb') as f: pickle.dump(xgb, f)
with open('scaler.pkl',    'wb') as f: pickle.dump(scaler, f)
with open('feature_cols.json', 'w') as f: json.dump(FEAT, f, indent=2)

# Write evaluation report
report_lines = [
    'ClinicalGuard — Model Evaluation Report',
    '=' * 55,
    f'Dataset: clinicalguard_realistic_dataset.csv',
    f'Train rows: {len(X_train)} | Test rows: {len(X_test)}',
    f'Class balance: {dict(y.value_counts().sort_index())}',
    '',
]
for key, res in results.items():
    report_lines += [
        f"Model: {res['name']}",
        f"  Accuracy       : {res['acc']*100:.2f}%",
        f"  ROC-AUC        : {res['roc']:.4f}",
        f"  MANIP Recall   : {res['rec_m']*100:.2f}%",
        f"  MANIP Precision: {res['prec_m']*100:.2f}%",
        f"  MANIP F1       : {res['f1_m']*100:.2f}%",
        f"  AUTH  Recall   : {res['rec_a']*100:.2f}%",
        f"  FP (dangerous) : {res['fp']}",
        f"  FN (nuisance)  : {res['fn']}",
        f"  PASS >=85% recall: {'YES' if res['rec_m']>=0.85 else 'NO'}",
        '',
    ]
report_lines.append(f'Winner (best MANIP recall): {results[best_key]["name"]}')

with open('evaluation_report.txt', 'w') as f:
    f.write('\n'.join(report_lines))

print('Saved artefacts:')
for fname in ['dt_model.pkl','rf_model.pkl','xgb_model.pkl','scaler.pkl',
              'feature_cols.json','evaluation_report.txt',
              'feature_distributions.png','feature_importance.png','shap_summary.png']:
    if os.path.exists(fname):
        sz = os.path.getsize(fname)
        print(f'  {fname:35s}  {sz:>8,} bytes')

## Step 10 — Download everything

In [ ]:
from google.colab import files

download_files = [
    'dt_model.pkl',
    'rf_model.pkl',
    'xgb_model.pkl',
    'scaler.pkl',
    'feature_cols.json',
    'evaluation_report.txt',
    'feature_distributions.png',
    'feature_importance.png',
    'shap_summary.png',
]

for fname in download_files:
    if os.path.exists(fname):
        files.download(fname)
        print(f'Downloading: {fname}')
    else:
        print(f'Missing: {fname}')

print('\nDone! Place the .pkl files and feature_cols.json into:')
print('  ClinicalGuard/backend/models/')
print('Then restart the backend and it will use the new models.')